## **ESTABELECENDO CONEXÃO COM BANCO DE DADOS**
> ## Credenciais e criação da engine SQLAlchemy para acesso ao banco MySQL

In [ ]:
import pandas as pd
from sqlalchemy import create_engine, text

!pip install pymysql


# Criando dicionário para armazenar informações para string de conexao
# Credenciais de acesso ao banco de dados removidas por segurança – acesso via banco privado da instituição Harve
credenciais = {
"host": "xxx",
"user": "xxx",
"password": "xxx",
"porta": 3306,
"database": "xxx"
}

# Fazendo string de conexao
string_conexao = f'mysql+pymysql://{credenciais['user']}:{credenciais['password']}@{credenciais['host']}:{credenciais['porta']}/{credenciais['database']}'

# Criando engine para conectar no banco
engine = create_engine(string_conexao)

In [ ]:
# Carregar tabelas principais
tabelas = {
    'df_orders': 'olist_orders_dataset',
    'df_order_items': 'olist_order_items_dataset',
    'df_products': 'olist_products_dataset',
    'df_customers': 'olist_customers_dataset',
    'df_payments': 'olist_order_payments_dataset',
    'df_reviews': 'olist_order_reviews_dataset',
    'df_sellers': 'olist_sellers_dataset',
    'df_geolocation': 'olist_geolocation_dataset'
}

# Carregar cada dataframe
dfs = {}
for nome, tabela in tabelas.items():
    query = f"SELECT * FROM {tabela}"
    dfs[nome] = pd.read_sql(query, engine)

## **INICIANDO LIMPEZA E TRANSFORMAÇÃO**

In [ ]:
dfs['df_orders'].head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00


In [ ]:
dfs['df_orders'].dtypes

,0
order_id,object
customer_id,object
order_status,object
order_purchase_timestamp,object
order_approved_at,object
order_delivered_carrier_date,object
order_delivered_customer_date,object
order_estimated_delivery_date,object


In [ ]:
# Transformar colunas object em datatime: order_purchase_timestamp,	order_approved_at,	order_delivered_carrier_date,	order_delivered_customer_date,	order_estimated_delivery_date

In [ ]:
colunas_tratar_data = [
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
]

# Convertendo colunas de df_orders para datetime
for col in colunas_tratar_data:
    dfs['df_orders'][col] = pd.to_datetime(dfs['df_orders'][col], errors='coerce')

In [ ]:
# Verificando tratamento de colunas
dfs['df_orders'].dtypes

,0
order_id,object
customer_id,object
order_status,object
order_purchase_timestamp,datetime64[ns]
order_approved_at,datetime64[ns]
order_delivered_carrier_date,datetime64[ns]
order_delivered_customer_date,datetime64[ns]
order_estimated_delivery_date,datetime64[ns]


In [ ]:
# Extrair componentes para tabela
dfs['df_orders']['ano'] = dfs['df_orders']['order_purchase_timestamp'].dt.year
dfs['df_orders']['mes'] = dfs['df_orders']['order_purchase_timestamp'].dt.month
dfs['df_orders']['hora'] = dfs['df_orders']['order_purchase_timestamp'].dt.hour

In [ ]:
# Criando nova coluna chamada mes_ano no formato YYYY-MM a partir da coluna order_purchase_timestamp
dfs['df_orders']['mes_ano'] = dfs['df_orders']['order_purchase_timestamp'].dt.strftime('%Y-%m')

In [ ]:
# Verificando novas colunas 'mes', 'ano', 'hora, 'mes_ano'
dfs['df_orders'].columns

Index(['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp',
       'order_approved_at', 'order_delivered_carrier_date',
       'order_delivered_customer_date', 'order_estimated_delivery_date', 'ano',
       'mes', 'hora', 'mes_ano'],
      dtype='object')

In [ ]:
dfs['df_orders'].head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,ano,mes,hora,mes_ano
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,2017,10,10,2017-10
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,2018,7,20,2018-07
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,2018,8,8,2018-08
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,2017,11,19,2017-11
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,2018,2,21,2018-02


In [ ]:
# Analisando outras tabelas
dfs['df_order_items'].columns

Index(['order_id', 'order_item_id', 'product_id', 'seller_id',
       'shipping_limit_date', 'price', 'freight_value'],
      dtype='object')

In [ ]:
dfs['df_order_items'].dtypes

,0
order_id,object
order_item_id,int64
product_id,object
seller_id,object
shipping_limit_date,object
price,float64
freight_value,float64


In [ ]:
dfs['df_order_items'].head()


,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14


In [ ]:
# Tranformando coluna shipping_limit_date para datatime
dfs['df_order_items']['shipping_limit_date'] = pd.to_datetime(dfs['df_order_items']['shipping_limit_date'], errors='coerce')

In [ ]:
# Conferindo
dfs['df_order_items'].dtypes

,0
order_id,object
order_item_id,int64
product_id,object
seller_id,object
shipping_limit_date,datetime64[ns]
price,float64
freight_value,float64


In [ ]:
dfs['df_customers'].dtypes

,0
customer_id,object
customer_unique_id,object
customer_zip_code_prefix,int64
customer_city,object
customer_state,object


In [ ]:
# Verificando valores unicos em estados e cidades
print(dfs['df_customers']['customer_state'].unique())
print(dfs['df_customers']['customer_city'].unique()) # Caso fosse usar cidades precisaria de tratamento

['SP' 'SC' 'MG' 'PR' 'RJ' 'RS' 'PA' 'GO' 'ES' 'BA' 'MA' 'MS' 'CE' 'DF'
 'RN' 'PE' 'MT' 'AM' 'AP' 'AL' 'RO' 'PB' 'TO' 'PI' 'AC' 'SE' 'RR']
['franca' 'sao bernardo do campo' 'sao paulo' ... 'eugenio de castro'
 'Alegre Porto' 'florianópolia']


In [ ]:
dfs['df_payments'].dtypes

,0
order_id,object
payment_sequential,int64
payment_type,object
payment_installments,int64
payment_value,float64


In [ ]:
dfs['df_payments'].head()

,order_id,payment_sequential,payment_type,payment_installments,payment_value
0,b81ef226f3fe1789b1e8b2acac839d17,1,credit_card,8,99.33
1,a9810da82917af2d9aefd1278f1dcfa0,1,credit_card,1,24.39
2,25e8ea4e93396b6fa0d3dd708e76c1bd,1,credit_card,1,65.71
3,ba78997921bbcdc1373bb41e913ab953,1,credit_card,8,107.78
4,42fdf880ba16b47b59251dd489d4441a,1,credit_card,2,128.45


In [ ]:
# OBS: customer_unique_id é o que realmente identifica UMA pessoa. customer_id é só um identificador de pedido.

In [ ]:
dfs['df_orders'].dtypes # Caso fosse usar mes_ano, tranformaria em datatime

,0
order_id,object
customer_id,object
order_status,object
order_purchase_timestamp,datetime64[ns]
order_approved_at,datetime64[ns]
order_delivered_carrier_date,datetime64[ns]
order_delivered_customer_date,datetime64[ns]
order_estimated_delivery_date,datetime64[ns]
ano,int32
mes,int32


In [ ]:
dfs['df_orders'].head(10)

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,ano,mes,hora,mes_ano
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,2017,10,10,2017-10
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,2018,7,20,2018-07
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,2018,8,8,2018-08
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,2017,11,19,2017-11
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,2018,2,21,2018-02
5,a4591c265e18cb1dcee52889e2d8acc3,503740e9ca751ccdda7ba28e9ab8f608,delivered,2017-07-09 21:57:05,2017-07-09 22:10:13,2017-07-11 14:58:04,2017-07-26 10:57:55,2017-08-01,2017,7,21,2017-07
6,136cce7faa42fdb2cefd53fdc79a6098,ed0271e0b7da060a393796590e7b737a,invoiced,2017-04-11 12:22:08,2017-04-13 13:25:17,NaT,NaT,2017-05-09,2017,4,12,2017-04
7,6514b8ad8028c9f2cc2374ded245783f,9bdf08b4b3b52b5526ff42d37d47f222,delivered,2017-05-16 13:10:30,2017-05-16 13:22:11,2017-05-22 10:07:46,2017-05-26 12:55:51,2017-06-07,2017,5,13,2017-05
8,76c6e866289321a7c93b82b54852dc33,f54a9f0e6b351c431402b8461ea51999,delivered,2017-01-23 18:29:09,2017-01-25 02:50:47,2017-01-26 14:16:31,2017-02-02 14:08:10,2017-03-06,2017,1,18,2017-01
9,e69bfb5eb88e0ed6a785585b27e16dbf,31ad1d1b63eb9962463f764d4e6e0c9d,delivered,2017-07-29 11:55:02,2017-07-29 12:05:32,2017-08-10 19:45:24,2017-08-16 17:14:30,2017-08-23,2017,7,11,2017-07


In [ ]:
# Feito até agora:

# Carregamento e limpeza de dados

# Processamento e enrriquecimento dos dados atravez de novas colunas, métricas e agrupamentos

## **EXPORTANDO CSV TRATADOS**
># *Observação*: tabelas e colunas que não foram tratadas aqui foram tratadas diretamente no Power BI por meio do Power Query

In [ ]:
# Exportando para CSV
dfs['df_orders'].to_csv('olist_orders_tratado_dataset.csv', index=False, encoding='utf-8')

dfs['df_order_items'].to_csv('olist_order_items_tratado_dataset.csv', index=False, encoding='utf-8')

## **VALIDAÇÃO DE CONSISTÊNCIA COM O DATASET ORIGINAL:**
> # Utilizando Python para investigar valores

In [ ]:
# OBS: Ticket médio = Soma de todos os pagamentos por contagem de pedidos únicos (order_id unicos)

In [ ]:
# Df temporario para conferir vendas em sp
df_temp = pd.merge(
    dfs['df_customers'][['customer_id', 'customer_state']],
    dfs['df_orders'][['customer_id', 'order_id']],
    on='customer_id'
)

# orders → payments
df_temp = pd.merge(
    df_temp,
    dfs['df_payments'][['order_id', 'payment_value']],
    on='order_id'
)

# calculando sp
sp_total = df_temp[df_temp['customer_state'] == 'SP']['payment_value'].sum()

In [ ]:
sp_total

np.float64(5998226.96)

In [ ]:
# OBS:
# Total Pedidos = DISTINCTCOUNT(tabela[order_id])
# Total Clientes = DISTINCTCOUNT(tabela[customer_unique_id])

In [ ]:
# VERIFICANDO RECEITA (enquanto trabalhava com algumas tabelas no Power BI percebi alguns valores diferentes e vim conferir aqui)
# Verificando dados brutos
print("PAYMENT_VALUE - Verificação:")
print(f"Soma total: R$ {dfs['df_payments']['payment_value'].sum():,.2f}")
print(f"Média: R$ {dfs['df_payments']['payment_value'].mean():,.2f}")
print(f"Máximo: R$ {dfs['df_payments']['payment_value'].max():,.2f}")

# Fazendo merge e verificando se multiplicou
df_test = (dfs['df_customers']
           .merge(dfs['df_orders'], on='customer_id')
           .merge(dfs['df_payments'], on='order_id'))

print(f"\nApós merge com customers:")
print(f"Soma: R$ {df_test['payment_value'].sum():,.2f}")

# Contando linhas
print(f"\nContagem:")
print(f"Customers: {len(dfs['df_customers']):,}")
print(f"Orders: {len(dfs['df_orders']):,}")
print(f"Payments: {len(dfs['df_payments']):,}")

PAYMENT_VALUE - Verificação:
Soma total: R$ 16,008,872.12
Média: R$ 154.10
Máximo: R$ 13,664.08

Após merge com customers:
Soma: R$ 16,008,872.12

Contagem:
Customers: 99,443
Orders: 99,441
Payments: 103,886


In [ ]:
dfs['df_payments'].to_csv('olist_order_payments_teste_dataset.csv', index=False, encoding='utf-8')
# Transformei em CSV para fazer algumas comparações. Levei para o Power BI mas depois deletei

In [ ]:
# No Power BI a coluna de payments não estava formatada da forma correta, valores estavam multiplicados por 100, vim conferir aqui.
df_check = pd.read_csv('olist_order_payments_teste_dataset.csv')

print("=== INVESTIGAÇÃO FINAL ===")
print(f"Linhas: {len(df_check):,}")
print(f"Soma payment_value: R$ {df_check['payment_value'].sum():,.2f}")
print(f"Maior valor: R$ {df_check['payment_value'].max():,.2f}")
print(f"Média: R$ {df_check['payment_value'].mean():,.2f}")

# Resolução final:
# Valores normais no df original, o problema de multiplicação dos valores era apenas no Power BI, precisei trocar a localização para formatar corretamente a coluna

=== INVESTIGAÇÃO FINAL ===
Linhas: 103,886
Soma payment_value: R$ 16,008,872.12
Maior valor: R$ 13,664.08
Média: R$ 154.10


In [ ]:
dfs['df_payments'].head(10)

,order_id,payment_sequential,payment_type,payment_installments,payment_value
0,b81ef226f3fe1789b1e8b2acac839d17,1,credit_card,8,99.33
1,a9810da82917af2d9aefd1278f1dcfa0,1,credit_card,1,24.39
2,25e8ea4e93396b6fa0d3dd708e76c1bd,1,credit_card,1,65.71
3,ba78997921bbcdc1373bb41e913ab953,1,credit_card,8,107.78
4,42fdf880ba16b47b59251dd489d4441a,1,credit_card,2,128.45
5,298fcdf1f73eb413e4d26d01b25bc1cd,1,credit_card,2,96.12
6,771ee386b001f06208a7419e4fc1bbd7,1,credit_card,1,81.16
7,3d7239c394a212faae122962df514ac7,1,credit_card,3,51.84
8,1f78449c87a54faf9e96e88ba1491fa9,1,credit_card,6,341.09
9,0573b5e23cbd798006520e1d5b4c6714,1,boleto,1,51.95


In [ ]:
dfs['df_payments'].dtypes

,0
order_id,object
payment_sequential,int64
payment_type,object
payment_installments,int64
payment_value,float64


In [ ]:
dfs['df_payments'].to_csv('olist_order_payments_final_dataset.csv', index=False, decimal='.', sep=',', encoding='utf-8')

In [ ]:
# Juntando order_payments com customers por orders pois eles não tem relacionamento direto (ruim para usar no power bi)
payments_customers = (dfs['df_customers'][['customer_id', 'customer_state']]
           .merge(dfs['df_orders'][['customer_id', 'order_id']], on='customer_id')
           .merge(dfs['df_payments'][['order_id', 'payment_value']], on='order_id'))

# Exporte
payments_customers.to_csv('payments_customers.csv', index=False)

In [ ]:
# Criando tabelão para testar analise de cliente
analise_clientes = (dfs['df_customers']
           .merge(dfs['df_orders'], on='customer_id', how='left')
           .merge(dfs['df_order_items'], on='order_id', how='left')
           .merge(dfs['df_products'], on='product_id', how='left')
           .merge(dfs['df_payments'], on='order_id', how='left')
           .merge(dfs['df_reviews'], on='order_id', how='left')
           .merge(dfs['df_sellers'], on='seller_id', how='left'))

# Exportando para o Power BI, depois deletei
analise_clientes.to_csv('analise_clientes_completo_olist.csv', index=False, encoding='utf-8-sig')

In [ ]:
# Encontrando order_ids duplicados nas reviews
duplicados = dfs['df_reviews']['order_id'].value_counts()
print(f"Order_ids com mais de 1 review: {len(duplicados[duplicados > 1])}")

print(duplicados[duplicados > 1].head())

Order_ids com mais de 1 review: 544
order_id
73fc7af87114b39712e6da79b0a377eb    4
df56136b8031ecd28e200bb18e6ddb2e    3
c88b1d1b157a9999ce368f218a407141    3
03c939fd7fd3b38f8485a0f95798f1f6    3
8e17072ec97ce29f0e1f111e598b0c85    3
Name: count, dtype: int64


In [ ]:
# Contando pedidos por cliente DIRETO (sem merges complexos)
contagem_pedidos = (dfs['df_orders'][['customer_id', 'order_id']]
                    .merge(dfs['df_customers'][['customer_id', 'customer_unique_id']],
                           on='customer_id')
                    .groupby('customer_unique_id')
                    .agg(total_pedidos=('order_id', 'nunique'))
                    .reset_index())

# Verificando
print(f"Clientes: {len(contagem_pedidos):,}")
print(f"Máximo de pedidos: {contagem_pedidos['total_pedidos'].max()}")
print(f"Média de pedidos: {contagem_pedidos['total_pedidos'].mean():.2f}")

# Top 10 clientes com mais pedidos
top_10 = contagem_pedidos.nlargest(10, 'total_pedidos') # 10 maiores linhas de total_pedidos ordenadas de forma decrescente
print("\nTOP 10 clientes com mais pedidos:")
print(top_10.to_string(index=False))

Clientes: 96,096
Máximo de pedidos: 17
Média de pedidos: 1.03

TOP 10 clientes com mais pedidos:
              customer_unique_id  total_pedidos
8d50f5eadf50201ccdcedfb9e2ac8455             17
3e43e6105506432c953e165fb2acf44c              9
1b6c7548a2a1f9037c1fd3ddfed95f33              7
6469f99c1f9dfae7733b25662e7f1782              7
ca77025e7201e3b30c44b472ff346268              7
12f5d6e1cbf93dafd9dcc19095df0b3d              6
47c1a3033b8b77b3ab6e109eb4d5fdf3              6
63cfc61cee11cbe306bff5857d00bfe4              6
dc813062e0fc23409cd255f7f53c7074              6
de34b16117594161a6a89c50b289d35a              6


In [ ]:
df_pedidos_cliente = (dfs['df_orders'][['customer_id', 'order_id']]
                      .merge(dfs['df_customers'][['customer_id', 'customer_unique_id', 'customer_state']],
                             on='customer_id')
                      .groupby('customer_unique_id')
                      .agg({
                          'customer_state': 'first',
                          'order_id': 'nunique'
                      }).reset_index())


print(f"Maior qtd pedidos: {df_pedidos_cliente['order_id'].max()}")


df_metricas = (dfs['df_orders'][['customer_id', 'order_id']]
               .merge(dfs['df_customers'][['customer_id', 'customer_unique_id']], on='customer_id')
               .merge(dfs['df_payments'][['order_id', 'payment_value']], on='order_id', how='left')
               .merge(dfs['df_reviews'][['order_id', 'review_score']], on='order_id', how='left')
               .groupby('customer_unique_id')
               .agg({
                   'payment_value': 'sum',
                   'review_score': 'mean'
               }).reset_index())

df_final_correto = df_pedidos_cliente.merge(df_metricas, on='customer_unique_id', how='left')

print(f"\nClientes: {len(df_final_correto):,}")
print(f"Maior pedidos: {df_final_correto['order_id'].max()}")
print(f"Total gasto: R$ {df_final_correto['payment_value'].sum():,.2f}")

df_final_correto.to_csv('top_clientes_final.csv', index=False)

Maior qtd pedidos: 17

Clientes: 96,096
Maior pedidos: 17
Total gasto: R$ 16,082,649.40


In [ ]:
df_final_correto.to_csv('top_clientes_final.csv', index=False)
# Também doi exportado para comparações no Power BI e depois apagado